# Synthetic Data Simulation for Bayesian Price Modelling

This notebook contains the simulation code associated with the synthetic data analysis in the paper *Bayesian Asset Price Modelling with Asymmetric Mean Reversion*.

The aim is to simulate price paths from the Geometric Mean Reversion (GMR) model and fit the model back to the generated data using Bayesian inference through Stan and CmdStanPy. This provides a reproducibility check for the simulation section of the paper and illustrates whether the inference procedure can recover the true model parameters.

The notebook is written as a reproducible research companion. Some settings may be reduced for demonstration purposes so that the code can be run on a standard laptop. The full simulation settings used in the paper are more computationally expensive.


## 1. Imports

We first import the Python packages required for simulation, Bayesian estimation, and result handling. The main inference engine is Stan, accessed through the `cmdstanpy` interface.

In [ ]:
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
import scipy.stats as ss
from tqdm import tqdm
from IPython.display import clear_output

## 2. Stan model specification

The following Stan model defines the likelihood and priors for the GMR model. The parameters are the drift (\mu), volatility (\sigma), upward and downward reversion strengths (\lambda^+) and (\lambda^-), and the adaptive-average smoothing parameter (\alpha).

In the paper, the prior on (\alpha) is examined using Beta((1,b)) choices. For the final empirical specification, the paper uses a non-informative Beta((1,1)) prior.


In [ ]:
# 1) Define your Stan code as a Python string
stan_code = """
data {
  int<lower=0> N;
  real<lower=0> dt;
  array[N] real x;
}
parameters {
  real mu;
  real lambda_plus;
  real lambda_minus;
  real<lower=1e-9> sigma;
  real<lower=0,upper=1> alpha;
}
model {
  // Priors
  mu        ~ normal(0, 1);
  lambda_plus  ~ normal(0, 10);
  lambda_minus ~ normal(0, 10);
  sigma        ~ normal(0, 1);
  alpha    ~ beta(1,1);

  // Recursive EMA (internal only)
  vector[N] ema;
  ema[1] = x[1];
  ema[2] = x[2];
  for (i in 3:N)
    ema[i] = alpha * x[i-1] + (1 - alpha) * ema[i-2];

  // Likelihood
  for (n in 3:N) {
    real diff      = (ema[n-1] - x[n-1]) / x[n-1];
    real up_move   = fmax(diff, 0);
    real down_move = fmin(diff, 0);
    real theta        = x[n-1] + dt * (mu + lambda_plus * up_move + lambda_minus * down_move) * x[n-1];
    real sd        = fmax(1e-9, abs(x[n-1]) * sigma * sqrt(dt));
    x[n] ~ normal(theta, sd);
  }
}
"""

# 2) Write that string to a file called "model.stan" in your cwd
with open("beta_1_1.stan", "w") as f:
    f.write(stan_code)

# 3) Confirm the file exists
import os
print("Wrote to:", os.path.abspath("beta_1_5.stan"))
print("Directory now contains:", os.listdir(os.getcwd()))

In [ ]:
sm = CmdStanModel(stan_file="beta_1_5.stan")
print("Compiled executable:", sm.exe_file)

In [ ]:
def simulate_price(N,paths,dt,mu_sim,sigma_sim,lambda_plus_sim,lambda_minus_sim,alpha_sim):
    np.random.seed(seed=100)
    X_0_sim = 100
    X_sim = np.zeros((paths, N))
    EMA_sim = np.zeros((paths, N))
    DIFF_sim = np.zeros((paths, N))
    POS_DIFF_sim = np.zeros((paths, N))
    NEG_DIFF_sim = np.zeros((paths, N))

    X_sim[:, 0:2] = X_0_sim
    EMA_sim[:, 0:2] = X_0_sim
    DIFF_sim[:, 0:2] = 0
    POS_DIFF_sim[:, 0:2] = 0
    NEG_DIFF_sim[:, 0:2] = 0

    W = ss.norm.rvs(loc=0, scale=1, size=(paths, N))

    for t in range(1, N - 1):
        X_sim[:, t + 1] = X_sim[:, t] + X_sim[:, t] * np.sqrt(dt) * sigma_sim * (W[:, t]) + dt * (mu_sim + lambda_plus_sim * (POS_DIFF_sim[:, t]) + lambda_minus_sim * (NEG_DIFF_sim[:, t])) * X_sim[:, t]
        EMA_sim[:, t + 1] = alpha_sim * X_sim[:, t] + (1 - alpha_sim) * EMA_sim[:, t - 1]
        DIFF_sim[:, t + 1] = (EMA_sim[:, t + 1] - X_sim[:, t + 1]) / X_sim[:, t + 1]
        POS_DIFF_sim[:, t + 1] = np.where(DIFF_sim[:, t + 1] < 0, 0, DIFF_sim[:, t + 1])
        NEG_DIFF_sim[:, t + 1] = np.where(DIFF_sim[:, t + 1] > 0, 0, DIFF_sim[:, t + 1])

    df_data = pd.DataFrame()
    pairs = [f'sim_asset_{i}' for i in range(paths)]
    for i in range(paths):
        df_data[f'sim_asset_{i}'] = X_sim[i]
    return df_data

In [ ]:
[mu_sim,sigma_sim,lambda_plus_sim,lambda_minus_sim,alpha_sim] =  [-0.0005,0.01,0,0,0.8]


N = 2500
paths = 20
dt = 1

simulate_price(N,paths,dt,mu_sim,sigma_sim,lambda_plus_sim,lambda_minus_sim,alpha_sim)

In [ ]:
np.random.seed(seed=42)

list_models = [
 'beta_1_1.stan'
]

cases = [
    [-0.0005,0.01,0,0,0.8]
]

N = 2500
paths = 1
dt = 1
pairs = [f'sim_asset_{i}' for i in range(paths)]

for case in cases:
    df_data = simulate_price(N,paths,dt,case[0],case[1],case[2],case[3],case[4])

    for name in tqdm(list_models):
        sm = CmdStanModel(stan_file=name)
        sigma_df = pd.DataFrame()
        lambda_plus_df = pd.DataFrame()
        lambda_minus_df = pd.DataFrame()
        mu_df = pd.DataFrame()
        alpha_df = pd.DataFrame()

        for asset in tqdm(pairs):
            y = df_data[asset].dropna().tolist()
            data = {"N": len(y), "dt": dt, "x": y}

            fit = sm.sample(
                data=data,
                chains=2,
                parallel_chains=2,
                iter_warmup=250,
                iter_sampling=250
            )

            sigma_df       [asset] = fit.stan_variable("sigma")
            lambda_plus_df [asset] = fit.stan_variable("lambda_plus")
            lambda_minus_df[asset] = fit.stan_variable("lambda_minus")
            mu_df       [asset] = fit.stan_variable("mu")
            alpha_df   [asset] = fit.stan_variable("alpha")
            clear_output()
        
        mu_str = str(case[0]).replace('.', 'p')
        sigma_str = str(case[1]).replace('.', 'p')
        lambdap_str = str(case[2]).replace('.', 'p')
        lambdam_str = str(case[3]).replace('.', 'p')
        alpha_str = str(case[4]).replace('.', 'p')

        save_name = os.path.splitext(name)[0]
        lambda_plus_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_lambda_plus.csv')
        lambda_minus_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_lambda_minus.csv')
        mu_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_mu.csv')
        alpha_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_alpha.csv')
        sigma_df.to_csv(f'{N}_points_{mu_str}_{sigma_str}_{lambdap_str}_{lambdam_str}_{alpha_str}_{save_name}_sigma.csv')